In [ ]:
import pandas as pd 
import numpy as np 

df = pd.read_csv("/Users/nakul/Desktop/PHYS/Equity-Signal-/data/snapshots.csv")
print(df.shape)
df.head 

#load 

In [3]:
df_ticker = df[df["ticker"].notna()].copy()
print("Kept:", df_ticker.shape, "of", df.shape)

df_ticker.to_csv("/Users/nakul/Desktop/PHYS/Equity-Signal-/data/fundamentals_ticker_filtered.csv", index=False)

# filter to ticker only rows

Kept: (507246, 22) of (989340, 22)


In [4]:
df_ticker["snapshot_dt"] = pd.to_datetime(df_ticker["snapshot"])
yearly_snapshots = sorted(df_ticker.loc[df_ticker["snapshot_dt"].dt.month == 1, "snapshot"].unique())

df_yearly = df_ticker[df_ticker["snapshot"].isin(yearly_snapshots)].copy()
print(df_yearly.groupby("snapshot")["cik"].nunique())

#yearly slice

snapshot
2013-01-01    2115
2014-01-01    2204
2015-01-01    2347
2016-01-01    2457
2017-01-01    2545
2018-01-01    2662
2019-01-01    2821
2020-01-01    2969
2021-01-01    3163
2022-01-01    3557
2023-01-01    3749
2024-01-01    3900
2025-01-01    4132
2026-01-01    4491
Name: cik, dtype: int64


In [6]:
ratios = df_yearly.copy()

ratios["ROE"] = ratios["Earnings_ttm"] / ratios["Equity"]
ratios["ROA"] = ratios["Earnings_ttm"] / ratios["Assets"]
ratios["GrossMargin"] = ratios["GrossProfit_ttm"] / ratios["Revenue_ttm"]
ratios["DebtToEquity"] = ratios["Liabilities"] / ratios["Equity"]
ratios["AssetTurnover"] = ratios["Revenue_ttm"] / ratios["Assets"]
ratios["EPS"] = ratios["Earnings_ttm"] / ratios["Shares"]

ratio_cols = ["ROE", "ROA", "GrossMargin", "DebtToEquity", "AssetTurnover", "EPS"]
ratios[ratio_cols] = ratios[ratio_cols].replace([np.inf, -np.inf], np.nan)

ratios[ratio_cols].describe()

,ROE,ROA,GrossMargin,DebtToEquity,AssetTurnover,EPS
count,37278.000000,37967.000000,20575.000000,42187.000000,34512.000000,37331.000000
mean,-2.230950,2.182248,0.515301,5.312571,0.977734,221.472918
std,273.037700,279.115676,16.212785,379.769700,46.360495,11219.548083
min,-31837.000000,-3517.079717,-358.368421,-9653.371951,-5.689900,-475000.000000
25%,-0.087295,-0.044379,0.234048,0.393167,0.180111,-0.319219
50%,0.077686,0.016636,0.375350,1.125929,0.546485,0.902556
75%,0.161052,0.061622,0.585386,2.839340,1.019710,2.866605
max,21903.673986,36625.116456,2029.333333,68763.000000,8612.004559,570400.000000


In [7]:
ratios = df_yearly.copy()

print("Before filtering:", ratios.shape)

# Drop rows where denominators are non-positive or missing, since ratios
# are undefined or economically meaningless in those cases:
# - Equity <= 0 breaks ROE and DebtToEquity (insolvent / negative book value)
# - Assets <= 0 breaks ROA and AssetTurnover (shouldn't happen, but check)
# - Revenue_ttm <= 0 breaks GrossMargin (no meaningful margin with no revenue)
# - Shares <= 0 breaks EPS

ratios = ratios[
    (ratios["Equity"] > 0) &
    (ratios["Assets"] > 0) &
    (ratios["Revenue_ttm"] > 0) &
    (ratios["Shares"] > 0)
].copy()

print("After filtering:", ratios.shape)

ratios["ROE"] = ratios["Earnings_ttm"] / ratios["Equity"]
ratios["ROA"] = ratios["Earnings_ttm"] / ratios["Assets"]
ratios["GrossMargin"] = ratios["GrossProfit_ttm"] / ratios["Revenue_ttm"]
ratios["DebtToEquity"] = ratios["Liabilities"] / ratios["Equity"]
ratios["AssetTurnover"] = ratios["Revenue_ttm"] / ratios["Assets"]
ratios["EPS"] = ratios["Earnings_ttm"] / ratios["Shares"]

ratio_cols = ["ROE", "ROA", "GrossMargin", "DebtToEquity", "AssetTurnover", "EPS"]
ratios[ratio_cols].describe()

Before filtering: (43112, 23)
After filtering: (31278, 23)


,ROE,ROA,GrossMargin,DebtToEquity,AssetTurnover,EPS
count,31238.000000,31238.000000,18834.000000,31278.000000,31278.000000,31238.000000
mean,-1.153468,0.322746,0.520657,6.162124,0.717317,265.509784
std,227.483900,92.914775,16.842691,190.674463,0.760377,11733.690027
min,-31837.000000,-3517.079717,-358.368421,-0.989183,0.000011,-308000.000000
25%,-0.025821,-0.009673,0.235168,0.638489,0.182270,-0.098726
50%,0.083455,0.021971,0.374024,1.397353,0.545927,1.184920
75%,0.158314,0.065618,0.578868,3.466550,1.002620,3.127102
max,21903.673986,15976.289231,2029.333333,29586.000000,18.437423,570400.000000


In [8]:
# Winsorize: clip each ratio at its 1st/99th percentile.
# This keeps every row (no further data loss) but caps the influence
# of extreme values caused by tiny but positive denominators.

ratio_cols = ["ROE", "ROA", "GrossMargin", "DebtToEquity", "AssetTurnover", "EPS"]

for col in ratio_cols:
    lower = ratios[col].quantile(0.01)
    upper = ratios[col].quantile(0.99)
    ratios[col] = ratios[col].clip(lower=lower, upper=upper)
    print(f"{col}: clipped to [{lower:.4f}, {upper:.4f}]")

print("\nRatio summary after winsorizing:")
ratios[ratio_cols].describe()

ROE: clipped to [-4.3473, 1.2932]
ROA: clipped to [-1.2033, 0.3003]
GrossMargin: clipped to [-0.3059, 0.9908]
DebtToEquity: clipped to [0.0560, 32.5030]
AssetTurnover: clipped to [0.0024, 3.2972]
EPS: clipped to [-8.8898, 29.5006]

Ratio summary after winsorizing:


,ROE,ROA,GrossMargin,DebtToEquity,AssetTurnover,EPS
count,31238.000000,31238.000000,18834.000000,31278.000000,31278.000000,31238.000000
mean,-0.054255,-0.022643,0.410266,3.185702,0.702390,2.146323
std,0.670059,0.216227,0.246486,4.817633,0.654750,4.853960
min,-4.347311,-1.203322,-0.305879,0.056030,0.002414,-8.889835
25%,-0.025821,-0.009673,0.235168,0.638489,0.182270,-0.098726
50%,0.083455,0.021971,0.374024,1.397353,0.545927,1.184920
75%,0.158314,0.065618,0.578868,3.466550,1.002620,3.127102
max,1.293196,0.300333,0.990776,32.502958,3.297169,29.500617


In [11]:
import yfinance as yf
import pandas as pd
import time
import random

ratios_labeled = ratios[ratios["snapshot"] != "2026-01-01"].copy()

unique_tickers = ratios_labeled["ticker"].unique().tolist()

# TEMPORARY: sample 5% of tickers for a fast end-to-end test run.
random.seed(42)
sample_size = max(1, int(len(unique_tickers) * 0.05))
unique_tickers = random.sample(unique_tickers, sample_size)

# Filter the dataframe down to just these tickers too, so the downstream
# apply() step has something real to compute on
ratios_labeled = ratios_labeled[ratios_labeled["ticker"].isin(unique_tickers)].copy()

print(f"Unique tickers to fetch: {len(unique_tickers)}")
print(f"Rows in sampled ratios_labeled: {len(ratios_labeled)}")

all_prices = {}
failed_tickers = []

batch_size = 50
for i in range(0, len(unique_tickers), batch_size):
    batch = unique_tickers[i:i + batch_size]
    try:
        data = yf.download(batch, start="2013-01-01", end="2027-01-01",
                            progress=False, group_by="ticker", threads=True)
        for t in batch:
            try:
                if len(batch) == 1:
                    series = data["Close"]
                else:
                    series = data[t]["Close"]
                if isinstance(series, pd.DataFrame):
                    series = series.iloc[:, 0]
                all_prices[t] = series.astype(float)
            except (KeyError, Exception):
                failed_tickers.append(t)
    except Exception as e:
        failed_tickers.extend(batch)
    time.sleep(1)
    if i % 500 == 0:
        print(f"Fetched {i}/{len(unique_tickers)} tickers...")

print(f"\nSuccessfully fetched: {len(all_prices)}")
print(f"Failed/missing: {len(failed_tickers)}")

spy_prices = yf.download("SPY", start="2013-01-01", end="2027-01-01", progress=False)["Close"]

def price_on_or_after(price_series, target_date):
    target = pd.Timestamp(target_date)
    valid = price_series[price_series.index >= target]
    if len(valid) == 0:
        return None
    val = valid.iloc[0]
    if hasattr(val, "item"):
        val = val.item()
    return val

def compute_excess_return(row):
    ticker = row["ticker"]
    snap_date = pd.Timestamp(row["snapshot"])
    fwd_date = snap_date + pd.DateOffset(years=1)

    if ticker not in all_prices:
        return None

    p_now = price_on_or_after(all_prices[ticker], snap_date)
    p_fwd = price_on_or_after(all_prices[ticker], fwd_date)
    spy_now = price_on_or_after(spy_prices, snap_date)
    spy_fwd = price_on_or_after(spy_prices, fwd_date)

    if any(v is None for v in [p_now, p_fwd, spy_now, spy_fwd]) or p_now == 0 or spy_now == 0:
        return None

    raw_return = p_fwd / p_now - 1
    market_return = spy_fwd / spy_now - 1
    return raw_return - market_return

ratios_labeled["excess_return"] = ratios_labeled.apply(compute_excess_return, axis=1)

print(f"\nRows with a valid excess_return: {ratios_labeled['excess_return'].notna().sum()} "
      f"of {len(ratios_labeled)}")
print("\nExcess return summary:")
print(ratios_labeled["excess_return"].describe())

ratios_labeled.to_csv("/Users/nakul/Desktop/PHYS/Equity-Signal-/data/ratio_with_returns.csv", index=False)
print("\nSaved to ../data/ratios_with_returns.csv")

Unique tickers to fetch: 158
Rows in sampled ratios_labeled: 1406
Fetched 0/158 tickers...


Failed to get ticker 'MGNI' reason: Failed to perform, curl: (28) Operation timed out after 10001 milliseconds with 1110 out of 1193 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$MGNI: possibly delisted; no timezone found

1 Failed download:
['MGNI']: possibly delisted; no timezone found



Successfully fetched: 158
Failed/missing: 0

Rows with a valid excess_return: 1393 of 1406

Excess return summary:
count    1393.000000
mean        0.002366
std         0.950289
min        -1.220645
25%        -0.296327
50%        -0.076787
75%         0.138146
max        20.392214
Name: excess_return, dtype: float64

Saved to ../data/ratios_with_returns.csv


In [12]:
import yfinance as yf
import pandas as pd
import time

ratios_labeled = ratios[ratios["snapshot"] != "2026-01-01"].copy()

unique_tickers = ratios_labeled["ticker"].unique().tolist()
print(f"Unique tickers to fetch: {len(unique_tickers)}")
print(f"Rows in ratios_labeled: {len(ratios_labeled)}")

all_prices = {}
failed_tickers = []

batch_size = 50
for i in range(0, len(unique_tickers), batch_size):
    batch = unique_tickers[i:i + batch_size]
    try:
        data = yf.download(batch, start="2013-01-01", end="2027-01-01",
                            progress=False, group_by="ticker", threads=True)
        for t in batch:
            try:
                if len(batch) == 1:
                    series = data["Close"]
                else:
                    series = data[t]["Close"]
                if isinstance(series, pd.DataFrame):
                    series = series.iloc[:, 0]
                all_prices[t] = series.astype(float)
            except (KeyError, Exception):
                failed_tickers.append(t)
    except Exception as e:
        failed_tickers.extend(batch)
    time.sleep(1)
    if i % 500 == 0:
        print(f"Fetched {i}/{len(unique_tickers)} tickers...")

print(f"\nSuccessfully fetched: {len(all_prices)}")
print(f"Failed/missing: {len(failed_tickers)}")

spy_prices = yf.download("SPY", start="2013-01-01", end="2027-01-01", progress=False)["Close"]

def price_on_or_after(price_series, target_date):
    target = pd.Timestamp(target_date)
    valid = price_series[price_series.index >= target]
    if len(valid) == 0:
        return None
    val = valid.iloc[0]
    if hasattr(val, "item"):
        val = val.item()
    return val

def compute_excess_return(row):
    ticker = row["ticker"]
    snap_date = pd.Timestamp(row["snapshot"])
    fwd_date = snap_date + pd.DateOffset(years=1)

    if ticker not in all_prices:
        return None

    p_now = price_on_or_after(all_prices[ticker], snap_date)
    p_fwd = price_on_or_after(all_prices[ticker], fwd_date)
    spy_now = price_on_or_after(spy_prices, snap_date)
    spy_fwd = price_on_or_after(spy_prices, fwd_date)

    if any(v is None for v in [p_now, p_fwd, spy_now, spy_fwd]) or p_now == 0 or spy_now == 0:
        return None

    raw_return = p_fwd / p_now - 1
    market_return = spy_fwd / spy_now - 1
    return raw_return - market_return

ratios_labeled["excess_return"] = ratios_labeled.apply(compute_excess_return, axis=1)

print(f"\nRows with a valid excess_return: {ratios_labeled['excess_return'].notna().sum()} "
      f"of {len(ratios_labeled)}")
print("\nExcess return summary:")
print(ratios_labeled["excess_return"].describe())

ratios_labeled.to_csv("/Users/nakul/Desktop/PHYS/Equity-Signal-/data/ratio_with_returns.csv", index=False)
print("\nSaved to ../data/ratios_with_returns.csv")

Unique tickers to fetch: 3161
Rows in ratios_labeled: 28331
Fetched 0/3161 tickers...


$CTA-P: possibly delisted; no timezone found

1 Failed download:
['CTA-P']: possibly delisted; no timezone found
$KELY: possibly delisted; no timezone found

1 Failed download:
['KELY']: possibly delisted; no timezone found
$SENE: possibly delisted; no timezone found

1 Failed download:
['SENE']: possibly delisted; no timezone found
$VLGE: possibly delisted; no timezone found

1 Failed download:
['VLGE']: possibly delisted; no timezone found
$SNFC: possibly delisted; no timezone found

1 Failed download:
['SNFC']: possibly delisted; no timezone found
$BELF: possibly delisted; no price data found  (1d 2013-01-01 -> 2027-01-01)
$CDR-P: possibly delisted; no timezone found

2 Failed downloads:
['BELF']: possibly delisted; no price data found  (1d 2013-01-01 -> 2027-01-01)
['CDR-P']: possibly delisted; no timezone found


Fetched 500/3161 tickers...


$FCNC: possibly delisted; no timezone found

1 Failed download:
['FCNC']: possibly delisted; no timezone found
$ARTN: possibly delisted; no timezone found

1 Failed download:
['ARTN']: possibly delisted; no timezone found
$RBCA: possibly delisted; no timezone found

1 Failed download:
['RBCA']: possibly delisted; no timezone found
$RUSH: possibly delisted; no price data found  (1d 2013-01-01 -> 2027-01-01)

1 Failed download:
['RUSH']: possibly delisted; no price data found  (1d 2013-01-01 -> 2027-01-01)


Fetched 1000/3161 tickers...


...

Fetched 2500/3161 tickers...


$ICR-P: possibly delisted; no timezone found

1 Failed download:
['ICR-P']: possibly delisted; no timezone found


Fetched 3000/3161 tickers...


$LTRY: possibly delisted; no timezone found

1 Failed download:
['LTRY']: possibly delisted; no timezone found
$BATR: possibly delisted; no timezone found

1 Failed download:
['BATR']: possibly delisted; no timezone found



Successfully fetched: 3161
Failed/missing: 0

Rows with a valid excess_return: 27742 of 28331

Excess return summary:
count    27742.000000
mean        -0.012829
std          0.762654
min         -5.978465
25%         -0.297154
50%         -0.077585
75%          0.151063
max         45.885125
Name: excess_return, dtype: float64

Saved to ../data/ratios_with_returns.csv


In [13]:
# Winsorize the target variable too, same reasoning as the X features:
# keep every row, but cap extreme tail outcomes so they don't dominate the loss.

lower = ratios_labeled["excess_return"].quantile(0.01)
upper = ratios_labeled["excess_return"].quantile(0.99)

print(f"excess_return clipped to [{lower:.4f}, {upper:.4f}]")

ratios_labeled["excess_return"] = ratios_labeled["excess_return"].clip(lower=lower, upper=upper)

print("\nexcess_return summary after winsorizing:")
print(ratios_labeled["excess_return"].describe())

# Re-save, now that the target is winsorized too
ratios_labeled.to_csv("/Users/nakul/Desktop/PHYS/Equity-Signal-/data/ratio_with_returns.csv", index=False)
print("\nSaved to ../data/ratios_with_returns.csv")

excess_return clipped to [-1.0005, 2.0698]

excess_return summary after winsorizing:
count    27742.000000
mean        -0.035154
std          0.468269
min         -1.000521
25%         -0.297154
50%         -0.077585
75%          0.151063
max          2.069841
Name: excess_return, dtype: float64

Saved to ../data/ratios_with_returns.csv
